# Cloud Interpolation Exploration

This notebook has two goals:

1. Start from a global patch id (`PATCH_ID = 555`), estimate a cloud mask, and compare interpolation methods.
2. Find all patches with cloud coverage above a threshold (`CLOUD_THRESHOLD_PCT = 20`).


In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.interpolate import griddata

plt.rcParams['figure.figsize'] = (7, 6)
plt.rcParams['image.cmap'] = 'viridis'

PROJECT_ROOT = Path.cwd().resolve()
RAW_PATCHES_DIR = (PROJECT_ROOT.parent / 'data' / 'river_patches').resolve()
METADATA_PATH = RAW_PATCHES_DIR / 'metadata.json'

PATCH_ID = 555
CLOUD_THRESHOLD_PCT = 20.0

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_PATCHES_DIR =', RAW_PATCHES_DIR)
print('METADATA_PATH =', METADATA_PATH)


In [ ]:
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

meta_by_id = {int(item['patch_id']): item for item in metadata}
print('metadata rows:', len(meta_by_id))

def resolve_patch_path(patch_id: int) -> Path:
    item = meta_by_id.get(int(patch_id))
    if item is None:
        raise KeyError(f'patch_id={patch_id} not found in metadata')
    rel_or_abs = Path(item['file'])
    if rel_or_abs.is_absolute():
        return rel_or_abs
    # Metadata stores values like river_patches/patch_123.tif -> normalize into data/
    if rel_or_abs.parts and rel_or_abs.parts[0] == 'river_patches':
        return PROJECT_ROOT.parent / 'data' / rel_or_abs
    return PROJECT_ROOT.parent / rel_or_abs

def read_base_stack(path: Path, n_bands: int = 5):
    with rasterio.open(path) as src:
        count = min(src.count, n_bands)
        stack = src.read(list(range(1, count + 1))).astype(np.float32)
        nodata = src.nodata
        profile = src.profile
    return stack, nodata, profile

def compute_cloud_mask(stack: np.ndarray, nodata=None, zero_eps: float = 1e-8):
    # stack shape: (bands, H, W)
    invalid = np.any(~np.isfinite(stack), axis=0)
    if nodata is not None:
        invalid |= np.any(np.isclose(stack, nodata), axis=0)
    # Earth Engine masked regions often arrive as all-zero vectors across bands.
    invalid |= np.all(np.abs(stack) <= zero_eps, axis=0)
    return invalid

def normalize_for_plot(img):
    x = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    p2, p98 = np.percentile(x, (2, 98))
    if p98 - p2 == 0:
        return np.zeros_like(x)
    return np.clip((x - p2) / (p98 - p2), 0, 1)


In [ ]:
patch_path = resolve_patch_path(PATCH_ID)
stack, nodata, profile = read_base_stack(patch_path, n_bands=5)
cloud_mask = compute_cloud_mask(stack, nodata=nodata)
cloud_pct = 100.0 * cloud_mask.mean()

print('patch:', patch_path)
print('stack shape:', stack.shape)
print('nodata:', nodata)
print(f'computed cloud coverage: {cloud_pct:.2f}%')

# pseudo RGB for this 5-band stack (band1=Green, band2=Red, band3=NIR)
green = stack[0]
red = stack[1]
nir = stack[2]
pseudo_rgb = np.dstack([normalize_for_plot(red), normalize_for_plot(green), normalize_for_plot(nir)])

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(pseudo_rgb)
ax[0].set_title(f'Patch {PATCH_ID} pseudo RGB')
ax[0].axis('off')

ax[1].imshow(cloud_mask, cmap='gray')
ax[1].set_title(f'Computed cloud mask ({cloud_pct:.2f}%)')
ax[1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
def interpolate_nearest(img: np.ndarray, missing_mask: np.ndarray):
    if not missing_mask.any():
        return img.copy()
    idx = ndimage.distance_transform_edt(missing_mask, return_distances=False, return_indices=True)
    return img[tuple(idx)]

def interpolate_linear(img: np.ndarray, missing_mask: np.ndarray):
    h, w = img.shape
    yy, xx = np.indices((h, w))
    known = ~missing_mask
    pts = np.column_stack([yy[known], xx[known]])
    vals = img[known]
    q_pts = np.column_stack([yy[missing_mask], xx[missing_mask]])

    out = img.copy()
    if len(q_pts) == 0:
        return out

    interp_vals = griddata(pts, vals, q_pts, method='linear')
    out[missing_mask] = interp_vals

    # Fill outside convex hull with nearest.
    still_nan = ~np.isfinite(out)
    if still_nan.any():
        out = interpolate_nearest(np.nan_to_num(out, nan=0.0), still_nan)
    return out

def interpolate_local_mean(img: np.ndarray, missing_mask: np.ndarray, kernel_size: int = 7, max_iter: int = 25):
    out = img.copy()
    out[missing_mask] = np.nan
    kernel = np.ones((kernel_size, kernel_size), dtype=np.float32)

    for _ in range(max_iter):
        nan_mask = np.isnan(out)
        if not nan_mask.any():
            break
        val_sum = ndimage.convolve(np.nan_to_num(out, nan=0.0), kernel, mode='nearest')
        val_cnt = ndimage.convolve((~np.isnan(out)).astype(np.float32), kernel, mode='nearest')
        est = np.where(val_cnt > 0, val_sum / np.maximum(val_cnt, 1e-6), np.nan)
        out[nan_mask] = est[nan_mask]

    still_nan = np.isnan(out)
    if still_nan.any():
        out = interpolate_nearest(np.nan_to_num(out, nan=0.0), still_nan)
    return out


In [ ]:
# Compare interpolation methods on one target band (NIR = band index 2)
target_band = stack[2].copy()

# Build evaluation holdout from valid pixels so we can measure reconstruction error.
rng = np.random.default_rng(42)
valid_pixels = ~cloud_mask
valid_coords = np.column_stack(np.where(valid_pixels))
n_eval = max(500, int(0.10 * len(valid_coords)))
n_eval = min(n_eval, len(valid_coords))
chosen = valid_coords[rng.choice(len(valid_coords), size=n_eval, replace=False)]

eval_mask = np.zeros_like(cloud_mask, dtype=bool)
eval_mask[chosen[:, 0], chosen[:, 1]] = True
work_mask = cloud_mask | eval_mask

band_for_interp = target_band.copy()
band_for_interp[work_mask] = 0.0

results = {}
timings = {}
for name, fn in {
    'nearest': interpolate_nearest,
    'linear': interpolate_linear,
    'local_mean': interpolate_local_mean,
}.items():
    t0 = time.time()
    if name == 'local_mean':
        filled = fn(band_for_interp, work_mask, kernel_size=7, max_iter=30)
    else:
        filled = fn(band_for_interp, work_mask)
    timings[name] = time.time() - t0
    results[name] = filled

true_vals = target_band[eval_mask]
metrics = []
for name, filled in results.items():
    pred = filled[eval_mask]
    mae = np.mean(np.abs(pred - true_vals))
    rmse = np.sqrt(np.mean((pred - true_vals) ** 2))
    metrics.append({'method': name, 'mae': mae, 'rmse': rmse, 'seconds': timings[name]})

metrics_df = pd.DataFrame(metrics).sort_values('mae')
display(metrics_df)

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(normalize_for_plot(target_band), cmap='magma')
ax[0].set_title('Original NIR')
ax[0].axis('off')

ax[1].imshow(work_mask, cmap='gray')
ax[1].set_title('Cloud + eval holdout mask')
ax[1].axis('off')

best_name = metrics_df.iloc[0]['method']
ax[2].imshow(normalize_for_plot(results[best_name]), cmap='magma')
ax[2].set_title(f'Best filled ({best_name})')
ax[2].axis('off')

diff = np.abs(results[best_name] - target_band)
im = ax[3].imshow(normalize_for_plot(diff), cmap='inferno')
ax[3].set_title('Absolute difference')
ax[3].axis('off')
plt.colorbar(im, ax=ax[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


## Part 2: find patches above cloud threshold

In [ ]:
rows = []
for patch_id, meta in meta_by_id.items():
    p = resolve_patch_path(patch_id)
    if not p.exists():
        continue
    stack_i, nodata_i, _ = read_base_stack(p, n_bands=5)
    mask_i = compute_cloud_mask(stack_i, nodata=nodata_i)
    cloud_pct_i = float(mask_i.mean() * 100.0)
    rows.append(
        {
            'patch_id': patch_id,
            'cloud_pct': cloud_pct_i,
            'file': str(p),
            'scene_cloud_cover': meta.get('cloud_cover', np.nan),
            'satellite': meta.get('satellite', 'UNKNOWN'),
        }
    )

cloud_df = pd.DataFrame(rows).sort_values('cloud_pct', ascending=False).reset_index(drop=True)
high_cloud_df = cloud_df[cloud_df['cloud_pct'] >= CLOUD_THRESHOLD_PCT].copy()

print(f'total scanned patches: {len(cloud_df)}')
print(f'patches with cloud >= {CLOUD_THRESHOLD_PCT:.1f}%: {len(high_cloud_df)}')
print('\nTop 20 cloudiest patches:')
display(high_cloud_df.head(20))

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.hist(cloud_df['cloud_pct'], bins=40, edgecolor='k', alpha=0.8)
ax.axvline(CLOUD_THRESHOLD_PCT, color='red', linestyle='--', linewidth=2, label=f'{CLOUD_THRESHOLD_PCT:.1f}% threshold')
ax.set_title('Distribution of computed cloud coverage across patches')
ax.set_xlabel('Cloud coverage (%)')
ax.set_ylabel('Patch count')
ax.legend()
plt.tight_layout()
plt.show()

out_csv = PROJECT_ROOT / 'data' / f'patches_cloud_over_{int(CLOUD_THRESHOLD_PCT)}pct.csv'
high_cloud_df.to_csv(out_csv, index=False)
print('Saved:', out_csv)
